# 📊 Employee Performance Predictor
## Interactive Notebook — EDA + Model Training

**Project:** Employee Performance Prediction using Data Analytics  
**Author:** [Your Name]  
**Date:** 2024  

---
### 🗂️ Table of Contents
1. Setup & Imports
2. Dataset Generation
3. Data Overview & Cleaning
4. Exploratory Data Analysis (EDA)
5. Feature Engineering
6. Model Training & Comparison
7. Evaluation & Results
8. Single Employee Prediction Demo
9. HR Insights Summary

## 1. Setup & Imports

In [ ]:
import sys, os
sys.path.insert(0, '..')  # allow importing from src/

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

%matplotlib inline
sns.set_theme(style='whitegrid', palette='muted')
print('✅ Setup complete')

## 2. Dataset Generation

In [ ]:
from src.generate_data import generate_dataset

df_raw = generate_dataset(n=1000, save_path='../data/employee_data.csv')
df_raw.head(10)

In [ ]:
print(f'Shape: {df_raw.shape}')
print('\nColumn Types:')
print(df_raw.dtypes)
print('\nMissing Values:')
print(df_raw.isnull().sum())

## 3. Data Overview & Cleaning

In [ ]:
from src.preprocess import clean_data, engineer_features, encode_features, get_features_target, scale_features

df = clean_data(df_raw.copy())
print(df.describe().round(2))

In [ ]:
# Performance label distribution
colors = {'Low': '#e74c3c', 'Medium': '#f39c12', 'High': '#27ae60'}
vc = df['performance_label'].value_counts().reindex(['Low','Medium','High'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
vc.plot.bar(color=[colors[l] for l in vc.index], ax=axes[0], edgecolor='white')
axes[0].set_title('Performance Label Count', fontweight='bold')
axes[0].set_ylabel('Employees')
axes[0].tick_params(axis='x', rotation=0)

axes[1].pie(vc, labels=vc.index, colors=[colors[l] for l in vc.index],
            autopct='%1.1f%%', startangle=140)
axes[1].set_title('Performance Share', fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Exploratory Data Analysis

In [ ]:
# Correlation heatmap
numeric_cols = df.select_dtypes(include=[np.number]).drop(columns=['employee_id'], errors='ignore')
corr = numeric_cols.corr()

fig, ax = plt.subplots(figsize=(14, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, ax=ax, annot_kws={'size': 8}, vmin=-1, vmax=1)
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Key factors comparison
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
order = ['Low', 'Medium', 'High']
palette = [colors[l] for l in order]

features = ['training_hours', 'absenteeism_days', 'projects_completed',
            'manager_rating', 'employee_satisfaction', 'overtime_hours']

for ax, feat in zip(axes.flatten(), features):
    sns.boxplot(data=df, x='performance_label', y=feat, order=order, palette=palette, ax=ax)
    ax.set_title(feat.replace('_', ' ').title(), fontweight='bold')
    ax.set_xlabel('')

plt.suptitle('HR Factors by Performance Level', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Department performance heatmap
dept_perf = df.groupby(['department', 'performance_label']).size().unstack(fill_value=0)
dept_perf_pct = dept_perf.div(dept_perf.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(dept_perf_pct[['Low','Medium','High']], annot=True, fmt='.1f',
            cmap='RdYlGn', linewidths=0.5, ax=ax)
ax.set_title('% Performance Level per Department', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Feature Engineering

In [ ]:
df = engineer_features(df)
df, encoders = encode_features(df, encoder_path='../models/label_encoders.pkl')

print('New engineered features:')
new_cols = ['salary_per_exp_year','productivity_index','engagement_score',
            'overwork_flag','high_absenteeism','job_level_num']
df[new_cols].describe().round(3)

## 6. Model Training & Comparison

In [ ]:
from src.preprocess import FEATURE_COLS

X, y = get_features_target(df)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                     random_state=42, stratify=y)
X_train_s, X_test_s, scaler = scale_features(X_train, X_test,
                                               scaler_path='../models/scaler.pkl')

print(f'Train: {X_train_s.shape}  |  Test: {X_test_s.shape}')

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score
import joblib

models = {
    'Logistic Regression' : LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest'       : RandomForestClassifier(n_estimators=200, random_state=42),
    'Gradient Boosting'   : GradientBoostingClassifier(n_estimators=150, random_state=42),
}

results = {}
for name, mdl in models.items():
    mdl.fit(X_train_s, y_train)
    acc = accuracy_score(y_test, mdl.predict(X_test_s))
    results[name] = acc
    print(f'{name:25s}  Accuracy: {acc:.4f}')

best_name = max(results, key=results.get)
best_model = models[best_name]
print(f'\n🏆 Best: {best_name}  ({results[best_name]:.4f})')
joblib.dump(best_model, '../models/best_model.pkl')

In [ ]:
# Model comparison bar chart
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(results.keys(), results.values(),
               color=['#3498db','#e74c3c','#2ecc71'], edgecolor='white', linewidth=1.5)
ax.set_ylim(0, 1)
ax.set_ylabel('Test Accuracy')
ax.set_title('Model Comparison — Test Accuracy', fontweight='bold')
for bar, val in zip(bars, results.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.4f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Evaluation & Results

In [ ]:
y_pred = best_model.predict(X_test_s)
LABEL_NAMES = ['Low', 'Medium', 'High']

print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=LABEL_NAMES))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(7, 5))
ConfusionMatrixDisplay(cm, display_labels=LABEL_NAMES).plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title(f'Confusion Matrix — {best_name}', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    idx = np.argsort(importances)[::-1]
    
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(range(len(FEATURE_COLS)), importances[idx],
           color=plt.cm.viridis(np.linspace(0.2, 0.9, len(FEATURE_COLS))))
    ax.set_xticks(range(len(FEATURE_COLS)))
    ax.set_xticklabels([FEATURE_COLS[i] for i in idx], rotation=45, ha='right', fontsize=9)
    ax.set_title('Feature Importance — Top Predictors', fontweight='bold')
    ax.set_ylabel('Importance')
    plt.tight_layout()
    plt.show()

## 8. Single Employee Prediction Demo

In [ ]:
from src.predict import predict_single, display_result

# 🧪 Change values here to test different employee profiles
employee = {
    'employee_id'          : 9999,
    'age'                  : 30,
    'gender'               : 'Female',
    'education_level'      : "Master's",
    'department'           : 'Engineering',
    'job_level'            : 'Senior',
    'years_experience'     : 7,
    'salary'               : 82000,
    'training_hours'       : 45,
    'absenteeism_days'     : 2,
    'projects_completed'   : 9,
    'overtime_hours'       : 8,
    'manager_rating'       : 4.5,
    'employee_satisfaction': 4.2,
    'years_since_promotion': 1,
}

result = predict_single(employee)
display_result(employee, result)

## 9. HR Insights Summary

In [ ]:
# Top insights from dataset
print('='*60)
print('  📋  KEY HR INSIGHTS FROM ANALYSIS')
print('='*60)

# Insight 1: Avg training hours by performance
avg_train = df.groupby('performance_label')['training_hours'].mean().reindex(['Low','Medium','High'])
print('\n1️⃣  Avg Training Hours by Performance:')
for lbl, val in avg_train.items():
    print(f'   {lbl:8s}: {val:.1f} hrs')

# Insight 2: Avg absenteeism
avg_abs = df.groupby('performance_label')['absenteeism_days'].mean().reindex(['Low','Medium','High'])
print('\n2️⃣  Avg Absenteeism Days by Performance:')
for lbl, val in avg_abs.items():
    print(f'   {lbl:8s}: {val:.1f} days')

# Insight 3: Dept with most High performers
top_dept = df[df['performance_label']=='High']['department'].value_counts().idxmax()
print(f'\n3️⃣  Department with Most High Performers: {top_dept}')

# Insight 4: Avg manager rating
avg_mgr = df.groupby('performance_label')['manager_rating'].mean().reindex(['Low','Medium','High'])
print('\n4️⃣  Avg Manager Rating by Performance:')
for lbl, val in avg_mgr.items():
    print(f'   {lbl:8s}: {val:.2f}')

print('\n✅ Notebook complete!')